# Laporan Mini Project
## Simulasi Link Budget dan Trade-off Modulasi pada Sistem Komunikasi Nirkabel

**Mata Kuliah:** Sistem Komunikasi Nirkabel  
**Tanggal:** 20 Mei 2026

**Kelompok C**

| No | Nama | NPM |
|:--:|------|-----|
| 1 | Muhammad Jibril Adrian | 2206059660 |
| 2 | Falah Andhesryo | 2306161990 |
| 3 | Muhammad Riyan Satrio Wibowo | 2306229323 |
| 4 | Ryan Adidaru Excel Barnabi | 2306266994 |

---

## 1. Pendahuluan

Perancangan sistem komunikasi nirkabel yang handal memerlukan pemahaman mendalam tentang *link budget* — sebuah perhitungan yang memodelkan semua faktor penguatan dan rugi-rugi sinyal dari pemancar hingga penerima. Mini project ini mensimulasikan sistem komunikasi untuk drone video monitoring yang beroperasi pada tiga kondisi lingkungan berbeda (perkotaan, pedesaan, dan pegunungan) dengan variasi frekuensi dan skema modulasi digital.

**Tujuan:**
1. Menghitung dan memvisualisasikan Free Space Path Loss (FSPL) untuk tiga frekuensi.
2. Melakukan analisis link budget lengkap (daya terima, noise floor, link margin).
3. Membandingkan performa BER vs Eb/N₀ untuk empat skema modulasi.
4. Memberikan rekomendasi desain berbasis data simulasi untuk tiga skenario deployment.

---

## 2. Dasar Teori

### 2.1 Free Space Path Loss (FSPL)

FSPL memodelkan rugi-rugi propagasi sinyal di ruang bebas tanpa hambatan fisik:

$$\text{FSPL}_{\text{dB}} = 32.44 + 20\log_{10}(f_{\text{MHz}}) + 20\log_{10}(d_{\text{km}})$$

Karena faktor $20\log_{10}(f)$, frekuensi tinggi mengalami path loss lebih besar pada jarak yang sama.

### 2.2 Link Budget

Daya terima $P_r$ dihitung dengan:

$$P_r = P_t + G_t + G_r - L_{\text{path}} - L_{\text{misc}} \quad [\text{dBm}]$$

Noise floor thermal:

$$N_{\text{floor}} = -174 + 10\log_{10}(B) + NF \quad [\text{dBm}]$$

*Link margin* (LM) mengukur cadangan daya di atas ambang sensitivitas:

$$LM = P_r - S_{\min} \quad [\text{dB}]$$

di mana $S_{\min} = N_{\text{floor}} + \text{SNR}_{\min}$.

### 2.3 BER dan Eb/N₀

Hubungan SNR dan Eb/N₀:

$$\text{SNR}_{\text{dB}} = \frac{E_b}{N_0}_{\text{dB}} + 10\log_{10}(\eta)$$

dengan $\eta = \log_2 M$ adalah spectral efficiency (bps/Hz).

BER untuk M-QAM dengan Gray coding:

$$\text{BER} \approx \frac{4}{\log_2 M}\left(1 - \frac{1}{\sqrt{M}}\right) Q\left(\sqrt{\frac{3\log_2 M \cdot E_b/N_0}{M-1}}\right)$$

---

## 3. Metodologi Simulasi

### Parameter Sistem

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

# System parameters
params = {
    'Pt (Tx Power)':         '30 dBm',
    'Gt (Tx Antenna Gain)':  '5 dBi',
    'Gr (Rx Antenna Gain)':  '5 dBi',
    'L_misc':                '3 dB',
    'Noise Figure (NF)':     '5 dB',
    'Frekuensi':             '700 MHz, 2.4 GHz, 28 GHz',
    'Bandwidth':             '1 MHz, 10 MHz, 100 MHz',
    'Modulasi':              'BPSK, QPSK, 16-QAM, 64-QAM',
    'Target BER':            '10⁻³',
    'Jarak simulasi':        '100 m – 10 km',
}

print('Parameter Sistem:')
for k, v in params.items():
    print(f'  {k:30s}: {v}')

---
## 4. Hasil dan Analisis

### 4.1 Bagian 1 — Free Space Path Loss

In [ ]:
from fspl import print_fspl_table, plot_fspl
print_fspl_table()
plot_fspl(save_dir='../plots')

In [ ]:
img = mpimg.imread('../plots/fspl_vs_distance.png')
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

#### Analisis FSPL

**Mengapa frekuensi tinggi memiliki path loss lebih besar?**

Dalam model FSPL, panjang gelombang $\lambda = c/f$. Antena isotropik memiliki aperture efektif $A_e = \lambda^2/(4\pi)$. Semakin tinggi frekuensi, semakin kecil $\lambda$, sehingga $A_e$ mengecil dan daya yang tertangkap lebih kecil. Pada jarak 1 km, perbedaan FSPL antara 700 MHz dan 28 GHz mencapai **32 dB** — perbedaan daya faktor ~1600×.

**Mengapa 700 MHz cocok untuk coverage luas?**

Pada 700 MHz, FSPL di jarak 10 km hanya 109 dB, dibanding 141 dB untuk 28 GHz. Selisih 32 dB ini berarti 700 MHz dapat menjangkau jarak ~40× lebih jauh dengan daya yang sama. Penetrasi terhadap gedung dan terrain juga lebih baik karena panjang gelombang lebih panjang (~43 cm vs ~1 mm).

**Trade-off utama mmWave (28 GHz):**

| Aspek | Keunggulan | Kelemahan |
|-------|------------|-----------|
| Bandwidth | Tersedia bandwidth sangat lebar (GHz) | Path loss sangat tinggi (+32 dB vs 700 MHz) |
| Coverage | Sel sangat kecil (< 200 m) untuk throughput tinggi | Tidak cocok untuk deployment luas |
| Penetrasi | — | Terblokir oleh tembok, pohon, hujan |
| Kapasitas | Throughput Gbps memungkinkan | Perlu line-of-sight |

---

### 4.2 Bagian 2 — Link Budget

In [ ]:
from link_budget import print_link_budget_table, plot_link_budget, noise_floor_dbm, BANDWIDTHS

print('=== Noise Floor per Bandwidth ===')
for bw_label, bw_hz in BANDWIDTHS.items():
    nf = noise_floor_dbm(bw_hz)
    print(f'  {bw_label:>10}: {nf:.1f} dBm')

print()
for f, label in [(700, '700 MHz'), (2400, '2.4 GHz'), (28000, '28 GHz')]:
    print_link_budget_table(f_mhz=f, d_km=1.0)

plot_link_budget(save_dir='../plots')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, fname in zip(axes, ['link_budget_received_power.png', 'link_margin_vs_distance.png']):
    img = mpimg.imread(f'../plots/{fname}')
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, fname in zip(axes, ['noise_floor_vs_bandwidth.png', 'link_margin_comparison.png']):
    img = mpimg.imread(f'../plots/{fname}')
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

#### Analisis Link Budget

**Pengaruh bandwidth terhadap noise floor:**

Setiap kenaikan bandwidth 10× menaikkan noise floor sebesar 10 dB (karena $N = -174 + 10\log_{10}B + NF$):

| Bandwidth | Noise Floor | Selisih vs 1 MHz |
|-----------|-------------|-------------------|
| 1 MHz     | -109 dBm    | —                 |
| 10 MHz    | -99 dBm     | +10 dB            |
| 100 MHz   | -89 dBm     | +20 dB            |

**Implikasi link margin:**

Pada 28 GHz dengan bandwidth 100 MHz, link margin di jarak 1 km sudah **negatif (-2.18 dB)** — artinya link tidak dapat beroperasi tanpa penguatan antena tambahan. Ini menunjukkan bahwa mmWave memerlukan antena terarah (beamforming) atau sel sangat kecil untuk compensate path loss yang tinggi.

**Trade-off bandwidth vs noise:**

Bandwidth lebar memungkinkan throughput data tinggi ($C = B\log_2(1+\text{SNR})$), namun noise floor yang lebih tinggi mengurangi link margin. Engineer harus menyeimbangkan kebutuhan throughput dengan coverage yang diinginkan.

---

### 4.3 Bagian 3 — Analisis BER vs Eb/N₀

In [ ]:
from modulation import print_modulation_summary, plot_ber, plot_snr_vs_throughput
print_modulation_summary()
plot_ber(save_dir='../plots')
plot_snr_vs_throughput(save_dir='../plots', bandwidth_hz=10e6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, fname in zip(axes, ['ber_vs_ebn0.png', 'ber_vs_snr.png']):
    img = mpimg.imread(f'../plots/{fname}')
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, fname in zip(axes, ['spectral_efficiency_tradeoff.png', 'throughput_vs_snr.png']):
    img = mpimg.imread(f'../plots/{fname}')
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

#### Analisis Modulasi

**Ringkasan parameter modulasi (target BER = 10⁻³):**

| Modulasi | η (bps/Hz) | Eb/N₀ min | SNR min  | Throughput @ 10 MHz |
|----------|-----------|-----------|---------|---------------------|
| BPSK     | 1         | 6.8 dB    | 6.8 dB  | 10 Mbps             |
| QPSK     | 2         | 6.8 dB    | 9.8 dB  | 20 Mbps             |
| 16-QAM   | 4         | 10.5 dB   | 16.5 dB | 40 Mbps             |
| 64-QAM   | 6         | 14.8 dB   | 22.5 dB | 60 Mbps             |

**Modulasi paling robust: BPSK**

BPSK hanya membutuhkan Eb/N₀ = 6.8 dB untuk mencapai BER = 10⁻³. Dengan konstelasi hanya 2 titik, jarak antar titik signal sangat besar sehingga noise margin maksimal. Sangat cocok untuk kondisi SNR rendah atau fluktuatif.

**Modulasi paling bandwidth-efficient: 64-QAM**

64-QAM memuat 6 bit per simbol, memberikan throughput 6× lebih tinggi dibanding BPSK pada bandwidth yang sama. Namun membutuhkan SNR ≥ 22.5 dB — 15.7 dB lebih tinggi dari BPSK. Trade-off ini membatasi penggunaannya pada kondisi SNR tinggi (jarak dekat, line-of-sight).

**Mengapa tidak selalu gunakan modulasi orde tertinggi?**

Grafik BER vs SNR menunjukkan bahwa pada SNR rendah (< 10 dB), 64-QAM memiliki BER mendekati 1 (hampir semua bit salah), sementara BPSK masih dapat beroperasi andal pada BER < 10⁻³. Dalam kondisi fading atau jarak jauh di mana SNR tidak terjamin tinggi, menggunakan 64-QAM justru menghasilkan throughput efektif nol karena error rate yang sangat tinggi.

---

## 5. Trade-off Engineering

### 5.1 Trade-off Frekuensi

| | 700 MHz | 2.4 GHz | 28 GHz |
|--|---------|---------|--------|
| FSPL @ 1 km | 89.3 dB | 100 dB | 121.4 dB |
| Coverage | Sangat luas | Sedang | Sangat terbatas |
| Penetrasi | Sangat baik | Baik | Buruk |
| Bandwidth tersedia | Terbatas | Sedang | Sangat lebar |
| Use case | Rural, IoT | Wi-Fi, LTE | 5G dense urban |

### 5.2 Trade-off Bandwidth vs SNR

Bandwidth lebar ↑ → Noise floor ↑ → Rx Sensitivity ↑ → Link Margin ↓

Ini bukan berarti bandwidth lebar selalu buruk — ia memungkinkan throughput lebih tinggi **jika** SNR yang tersedia mencukupi. Pada jarak dekat (SNR tinggi), bandwidth lebar menguntungkan. Pada jarak jauh (SNR rendah), bandwidth sempit menjaga link margin.

### 5.3 Trade-off Modulasi

Higher-order modulation (64-QAM) vs. robust modulation (BPSK):

| Aspek | BPSK | 64-QAM |
|-------|------|--------|
| Throughput | Rendah (1 bps/Hz) | Tinggi (6 bps/Hz) |
| Robustness | Sangat tinggi | Rendah |
| SNR minimum | 6.8 dB | 22.5 dB |
| Coverage | Luas | Terbatas |
| Latency | — | Memerlukan channel estimation akurat |

### 5.4 Bandwidth vs Power Efficiency

Berdasarkan Shannon Capacity: $C = B\log_2(1 + P/N_0 B)$

Untuk kapasitas konstan, menaikkan B sambil menurunkan $P$ memiliki batas — pada bandwidth sangat lebar, $P/N_0 B$ mendekati nol sehingga kapasitas tidak meningkat. Sebaliknya, BPSK/QPSK merupakan pendekatan yang lebih *power efficient*: throughput lebih rendah namun daya per bit ($E_b$) lebih kecil.

---

## 6. Rekomendasi Desain (Engineering Decision)

### Skenario A — Pedesaan

**Karakteristik:** Coverage luas, infrastruktur terbatas, user sedikit.

**Rekomendasi:**
- **Frekuensi:** 700 MHz — FSPL paling kecil, coverage terluas, penetrasi terrain baik.
- **Modulasi:** BPSK atau QPSK — tidak membutuhkan SNR tinggi, cocok untuk jarak jauh.
- **Bandwidth:** 1 MHz — noise floor -109 dBm, link margin maksimal.

**Justifikasi:** Pada 700 MHz dan BPSK dengan bandwidth 1 MHz, simulasi menunjukkan link margin > 35 dB bahkan di jarak 5 km. Ini memberikan cadangan yang cukup besar untuk mengatasi multipath fading dan shadowing di terrain pedesaan.

### Skenario B — Perkotaan

**Karakteristik:** User padat, throughput tinggi, interferensi tinggi.

**Rekomendasi:**
- **Frekuensi:** 2.4 GHz — menyeimbangkan coverage dan kapasitas spektral.
- **Modulasi:** 16-QAM (dalam kondisi sedang) atau 64-QAM (kondisi baik, jarak dekat).
- **Bandwidth:** 10–100 MHz — throughput tinggi dengan SNR perkotaan yang umumnya cukup.

**Justifikasi:** Di perkotaan, jarak antena-ke-user pendek (< 500 m), sehingga SNR tinggi dapat dicapai. Dengan 64-QAM @ 100 MHz bandwidth, throughput teoritis 600 Mbps per carrier. Link margin di 2.4 GHz, jarak 1 km, BW 10 MHz masih ~29 dB — memadai untuk sistem adaptif.

### Skenario C — Drone Pegunungan

**Karakteristik:** Fading berat, reliabilitas tinggi, SNR fluktuatif.

**Rekomendasi:**
- **Frekuensi:** 700 MHz — difraksi lebih baik di terrain kasar, less sensitive to rain fade.
- **Modulasi:** BPSK — paling robust, hanya butuh Eb/N₀ 6.8 dB, BER terendah pada SNR fluktuatif.
- **Bandwidth:** 1 MHz — noise floor minimal, link margin maksimal untuk menjaga koneksi saat SNR drop.

**Justifikasi:** Fading berat dapat menyebabkan SNR drop hingga 15–20 dB secara sementara. Dengan BPSK, sistem masih dapat beroperasi pada SNR serendah ~7 dB. Bandwidth sempit memastikan noise floor rendah (-109 dBm), memberikan buffer yang cukup saat deep fade terjadi.

---

## 7. Pembahasan Diskusi

**1. Mengapa 5G mmWave tidak cocok untuk coverage pedesaan?**

mmWave (24–100 GHz) memiliki FSPL sangat tinggi — pada 28 GHz, FSPL di 5 km mencapai 135 dB, sedangkan daya terima hanya -98 dBm yang hampir menyentuh noise floor bahkan dengan bandwidth sempit. Ditambah atenuasi atmosferik yang signifikan (hujan, oksigen) dan hambatan penetrasi terrain, mmWave hanya praktis untuk jarak < 200 m dalam line-of-sight. Pedesaan membutuhkan coverage 5–30 km per BTS, yang tidak mungkin dipenuhi mmWave.

**2. Mengapa komunikasi deep-space lebih mengutamakan power efficiency daripada bandwidth?**

Pada deep-space (Mars ~3×10⁸ km), path loss ~300 dB. Meningkatkan bandwidth hanya menaikkan noise floor, sementara daya pemancar sangat terbatas oleh panel surya. BPSK dengan FEC (Forward Error Correction) adalah pilihan karena mendekati Shannon limit pada SNR sangat rendah — yang jauh lebih kritis daripada throughput.

**3. Mengapa bandwidth besar tidak selalu menghasilkan performa lebih baik?**

Shannon Capacity: $C = B\log_2(1+\text{SNR})$. Jika SNR konstan, menggandakan B memang menggandakan C. Namun SNR tidak konstan — bandwidth lebih besar menaikkan noise power ($N = N_0 \cdot B$), sehingga SNR per hertz turun. Pada kondisi daya terbatas, ada titik optimal bandwidth yang memaksimalkan kapasitas.

**4. Mengapa engineer tidak selalu menggunakan modulasi orde tertinggi?**

64-QAM membutuhkan SNR ~22.5 dB untuk BER 10⁻³. Pada channel dengan SNR 15 dB (umum di outdoor mobile), 64-QAM memberikan BER > 0.1 — hampir semua bit error. Throughput efektif mendekati nol. Sistem adaptif modern (AMC — Adaptive Modulation and Coding) menyesuaikan modulasi secara real-time berdasarkan kondisi channel.

**5. Apa hubungan antara reliability, latency, dan coding redundancy?**

FEC (coding redundancy) meningkatkan reliability dengan menambahkan bit paritas, namun setiap bit tambahan mengurangi throughput efektif (code rate < 1). Untuk latency rendah (realtime control), overhead FEC harus minimal — artinya channel condition harus baik atau modulasi yang lebih robust dipilih. Dalam skenario drone pegunungan, tradeoff ini diselesaikan dengan memilih BPSK (sangat robust secara intrinsik) daripada QAM tinggi dengan heavy FEC.

---

## 8. Kesimpulan

1. **FSPL** berbanding lurus dengan kuadrat frekuensi: 28 GHz memiliki FSPL 32 dB lebih tinggi dari 700 MHz pada jarak sama. Frekuensi rendah cocok untuk coverage luas.

2. **Link budget** menunjukkan bahwa bandwidth lebih lebar meningkatkan noise floor 10 dB per dekade, langsung mengurangi link margin. Pada 28 GHz + 100 MHz bandwidth, link margin sudah negatif di 1 km tanpa antena terarah.

3. **Trade-off modulasi**: BPSK paling robust (SNR min 6.8 dB) tapi throughput rendah; 64-QAM throughput 6× lebih tinggi tapi butuh SNR 22.5 dB. Pilihan modulasi harus disesuaikan dengan kondisi channel.

4. **Rekomendasi per skenario:**
   - Pedesaan/drone pegunungan → 700 MHz + BPSK + BW sempit (prioritas coverage & reliability)
   - Perkotaan → 2.4 GHz + 16/64-QAM + BW lebar (prioritas kapasitas & throughput)

5. Simulasi ini membuktikan bahwa **tidak ada konfigurasi tunggal terbaik** — setiap parameter adalah trade-off yang harus diputuskan berdasarkan constraint deployment nyata.

---

## Referensi

1. Rappaport, T.S. (2002). *Wireless Communications: Principles and Practice* (2nd ed.). Prentice Hall.
2. Proakis, J.G. & Salehi, M. (2008). *Digital Communications* (5th ed.). McGraw-Hill.
3. Goldsmith, A. (2005). *Wireless Communications*. Cambridge University Press.
4. Haykin, S. & Moher, M. (2005). *Modern Wireless Communications*. Pearson.
5. ITU-R P.525-4 (2019). *Calculation of free-space attenuation*. International Telecommunication Union.
6. Shannon, C.E. (1948). A mathematical theory of communication. *Bell System Technical Journal*, 27(3), 379–423.